# ДЗ-2. Витрина клиента

Вы — аналитик ПРАЙМ, и у вас свой **сегмент клиентов**: его прислал бот 26 сентября, он же лежит в вашей строке `prime.assignments`, в колонке `client_filter`. Окно фактов у всех общее — первое полугодие 2026 года.

Задача недели — собрать **витрину клиента**: таблицу, в которой одна строка — один клиент, а рядом всё, что мы про него знаем. Из витрины потом считается всё остальное, поэтому лишние деньги в ней становятся лишними везде.

Идём по лестнице. Сначала вы дописываете один аргумент, как на семинаре, потом целую строку, потом пишете задачу сами, а в конце собираете витрину и доказываете протоколом, что двойного счёта нет.

### Как устроен ноутбук

| Часть | Что там | Что делаете вы |
|---|---|---|
| **1. Подготовка** | готовый код: подключение, ваш сегмент, выгрузка восьми таблиц | запускаете |
| **2. Уровень 1** | задачи 1–4: не хватает одного аргумента | вписываете аргумент вместо `ЗАПОЛНИТЕ` |
| **3. Уровень 2** | задачи 5–8: не хватает строки | дописываете строку |
| **4. Уровень 3** | задачи 9–12 | пишете код целиком |
| **5. Уровень 4** | задача 13: витрина и протокол сборки | пишете код целиком |
| **6. Итог** | готовый код: ваши ответы и проверка их формата | запускаете перед сдачей |
| **7. Исследование** | две части для тех, кто хочет 9 или 10 | по желанию |

После каждого уровня — вопрос, на который вы отвечаете словами в ячейке «✍️ Ответ на вопрос N».

**`ЗАПОЛНИТЕ`** — место, куда вписать ваш код. Пока оно не заполнено, ячейка падает с `NameError: name 'ЗАПОЛНИТЕ' is not defined` — так и задумано.

**Оценка.** Части 1–6 дают до 8 баллов: сделали всё аккуратно — 8, и это «отлично». 9 и 10 ставятся только за исследование в части 7. Все критерии и порядок сдачи — в [тексте задания](https://github.com/tikhomirovd/python-for-ba-hse-2026/tree/master/задания/дз-2-витрина-клиента).

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед сдачей выполните **Kernel → Restart Kernel and Run All Cells**: ноутбук должен пройти целиком без красного.

Ноутбук лежит в вашем репозитории как `notebooks/hw2.ipynb`, JupyterLab запускается из корня `prime-monitor` командой `uv run jupyter lab`.

## Часть 1. Подготовка — готовый код

Код в этой части писать не нужно: запустите ячейки по порядку.

| Ячейка | Что делает |
|---|---|
| 1.1 Подключение | подключается к базе через ваш `.env` и заводит функцию `q(sql)` |
| 1.2 Ваш сегмент | достаёт условие `client_filter` из вашей строки `prime.assignments` |
| 1.3 Выгрузка | загружает восемь таблиц — дальше всё в pandas |
| 1.4 Формат ответов | заводит служебную функцию для итога |

Весь SQL задания собран здесь, в готовом коде. Дальше вы работаете только в pandas.

### 1.1 Подключение

Функция `q(sql)` отправляет запрос в базу и возвращает `DataFrame`, как на семинарах. Соединение открывается на каждый запрос и сразу закрывается.

Если здесь ошибка `Пакет prime не найден`, ноутбук запущен не в окружении вашего репозитория. Если `Не найдена переменная PRIME_DSN`, в корне `prime-monitor` нет файла `.env`.

In [1]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

подключились как: avgavrilova_2


### 1.2 Ваш сегмент

Сегмент — это клиенты, которые встречаются в вашей таблице с вашим признаком за ваш период. Бот прислал его вам 26 сентября готовым условием для `where`, оно же лежит в вашей строке `prime.assignments`.

**Эту ячейку не меняйте.** При проверке преподаватель подставит сюда условие **другого** сегмента, и все ответы должны пересчитаться сами. Поэтому ниже нигде не вписывайте руками ни условие, ни числа своего сегмента.

In [2]:
CLIENT_FILTER = q("select client_filter from prime.assignments where login = current_user").iloc[0, 0]

print(CLIENT_FILTER)

client_id in (
    select s.owner_client_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.payment_method = 'balance'
      and p.paid_at >= '2025-07-01 00:00+00'
      and p.paid_at <  '2025-10-01 00:00+00'
)


### 1.3 Выгрузка

Восемь таблиц: первые пять — про ваш сегмент, последние три — справочник тарифов и данные всей базы.

| Переменная | Одна строка — это | Что внутри |
|---|---|---|
| `clients` | клиент сегмента | `client_id`, дата регистрации, регион, канал привлечения |
| `subs` | подписка, которой владеет клиент сегмента | тариф, даты начала и конца, пробная ли, причина отмены |
| `pay` | попытка списания по этим подпискам за окно фактов | сумма, **статус**, способ оплаты |
| `sub_members` | участник подписки из `subs` | `subscription_id` и `member_id` — кто подключён к подписке |
| `members` | участие клиента сегмента в любой подписке | `subscription_id` и `client_id` — клиент сегмента как участник |
| `tariffs` | цена тарифа в интервале дат | весь справочник, пять строк |
| `spend` | расходы канала за день | `channel`, `spend_date`, `amount` |
| `registrations` | канал × день регистрации | `n_clients` — сколько клиентов всей базы пришло из канала в этот день |

**Окно фактов общее для всех:** с 1 января по 30 июня 2026 года включительно. В запросе правый конец не включён: `paid_at < '2026-07-01'`.

Самые большие сегменты выгружаются около минуты и занимают в памяти до 2 ГБ — закройте лишние вкладки браузера. Если выгрузка идёт дольше двух минут, сервер оборвёт запрос — что делать, написано в тексте задания, раздел «Если что-то не работает».

In [3]:
WINDOW = ("2026-01-01", "2026-07-01")   # окно фактов: первое полугодие 2026, правый конец не включён
SEGMENT = f"select client_id from prime.clients where {CLIENT_FILTER}"

clients = q(f"""
    select client_id, registered_at, region_code, acquisition_channel
    from prime.clients
    where {CLIENT_FILTER}
    order by client_id
""")

subs = q(f"""
    select subscription_id, owner_client_id, tariff_id, started_at, ended_at, is_trial, cancel_reason
    from prime.subscriptions
    where owner_client_id in ({SEGMENT})
    order by subscription_id
""")

pay = q(f"""
    select p.payment_id, p.subscription_id, p.paid_at, p.amount, p.status, p.payment_method
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where s.owner_client_id in ({SEGMENT})
      and p.paid_at >= :a and p.paid_at < :b
    order by p.payment_id
""", a=WINDOW[0], b=WINDOW[1])

sub_members = q(f"""
    select m.subscription_id, m.client_id as member_id
    from prime.subscription_members m
    join prime.subscriptions s on s.subscription_id = m.subscription_id
    where s.owner_client_id in ({SEGMENT})
    order by m.subscription_id, m.client_id
""")

members = q(f"""
    select subscription_id, client_id
    from prime.subscription_members
    where client_id in ({SEGMENT})
    order by client_id, subscription_id
""")

tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")

spend = q("select channel, spend_date, amount from prime.marketing_spend order by channel, spend_date")

registrations = q("""
    select acquisition_channel as channel, registered_at::date as reg_date, count(*) as n_clients
    from prime.clients
    group by 1, 2
    order by 1, 2
""")

N = len(clients)
print("ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов:", N)
print()
for name, frame in [("clients", clients), ("subs", subs), ("pay", pay), ("sub_members", sub_members),
                    ("members", members), ("tariffs", tariffs), ("spend", spend),
                    ("registrations", registrations)]:
    print(f"{name:<14} {len(frame):>9} строк")

ЗЕРНО витрины: одна строка — один клиент сегмента. Клиентов: 25609

clients            25609 строк
subs               25609 строк
pay               114043 строк
sub_members        42661 строк
members            28523 строк
tariffs                5 строк
spend               3606 строк
registrations       6762 строк


### 1.4 Формат ответов

Служебная функция: по ней итог в части 6 проверяет, что ответ записан в нужном формате — целое, число с копейками, список. **Верно ли посчитано, она не знает и не сообщает**: это проверяет преподаватель. Код ячейки свёрнут, просто запустите её.

In [4]:
# Какой формат ответа ожидается в каждой задаче — человеческими словами.
EXPECTED = {
    1: "float до 2 знаков — рубли",
    2: "int",
    3: "int",
    4: "float до 2 знаков — рубли (0.0, если таких клиентов нет)",
    5: "float до 2 знаков — рубли",
    6: "float до 2 знаков — рубли",
    7: "int",
    8: "float до 1 знака — проценты",
    9: "int",
    10: "list [float до 2 знаков, int]",
    11: "list [float до 2 знаков, int]",
    12: "list из 3 int",
    13: "list из 3 int — то, что вернула print_protocol()",
}


def format_problem(n: int, answer) -> str | None:
    """Что не так с ФОРМАТОМ ответа задачи n (не с правильностью). None — всё в порядке."""

    def money(x) -> bool:   # float, у которого не больше двух знаков после запятой
        return type(x) is float and round(x, 2) == x

    def pct(x) -> bool:     # float, у которого не больше одного знака после запятой
        return type(x) is float and round(x, 1) == x

    def whole(x) -> bool:
        return type(x) is int

    if answer is None:
        return "не решена: answer_N = None, или ячейка задачи не запускалась, или упала"
    checks = {
        1: money, 2: whole, 3: whole, 4: money, 5: money, 6: money, 7: whole, 8: pct,
        9: whole,
        10: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        11: lambda a: type(a) is list and len(a) == 2 and money(a[0]) and whole(a[1]),
        12: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
        13: lambda a: type(a) is list and len(a) == 3 and all(whole(x) for x in a),
    }
    if checks[n](answer):
        return None
    return f"нужен {EXPECTED[n]}, сейчас {answer!r} ({type(answer).__name__})"


print("проверка формата готова")

проверка формата готова


## Часть 2. Уровень 1 — дыра в один аргумент

Как на семинаре: код готов, не хватает одного аргумента. Впишите его вместо `ЗАПОЛНИТЕ` и запустите ячейку.

Последняя строка каждой задачи кладёт ответ в `answer_N`. Эти строки и имена `answer_N` не меняйте: по ним работа проверяется автоматически, в том числе на другом сегменте.

**Не получилась задача — не оставляйте падающий код.** Ячейка с ошибкой останавливает Run All. Замените всё содержимое ячейки одной строкой `answer_N = None`, где N — номер задачи: итог покажет «❌ не решена», а остальное проверится как обычно.

**Ноутбук не скажет, правильно ли вы посчитали.** Итог в части 6 проверяет только формат. Правильность проверяет преподаватель — на вашем сегменте и на контрольном.

**В `pay` лежат все попытки списания:** `success`, `failed` и `refunded`. Какие из них деньги, решаете вы — это задача 1.

### Задача 1. Контрольное число ДЕНЬГИ

Первое правило сборки витрины: контрольное число считают до сборки. Мы заранее знаем, сколько денег принёс сегмент за окно, — и потом витрина обязана дать ровно столько же.

**Что сделать**

Оставьте в `pay` только платежи, которые действительно принесли деньги, и посчитайте их сумму.

**Формат ответа:** `float` до двух знаков, рубли — например `98765432.1`.

**Подсказка:** Какие статусы бывают, покажет `pay["status"].value_counts()`.

In [9]:
# Какие попытки списания — это деньги, которые остались у компании?
success = pay[pay["status"] == 'success']
revenue = success["amount"].sum()

print("платежей с этим статусом:", len(success))
answer_1 = round(float(revenue), 2)
answer_1

платежей с этим статусом: 108771


33275879.0

### Задача 2. Подписки без платежа

Руководитель спрашивает, сколько подписок сегмента за полгода не принесли ни рубля. Чтобы ответить, к каждой подписке присоединяют число её успешных платежей и смотрят, у кого пусто.

**Что сделать**

Выберите вид соединения так, чтобы подписки без платежей **остались** в результате — с пропуском в `n_payments`.

**Формат ответа:** `int` — например `4321`.

**Подсказка:** Слайд 7 Л4: `how=` решает судьбу строк, которым не нашлась пара.

In [10]:
# Сколько успешных платежей пришло по каждой подписке за окно.
by_sub = (pay[pay["status"] == "success"]
          .groupby("subscription_id", as_index=False)
          .agg(n_payments=("payment_id", "size")))

# Подписка без платежей должна остаться в результате — с пропуском в n_payments.
subs_pay = subs.merge(by_sub, on="subscription_id", how='left', validate="1:1")

no_payment = subs_pay["n_payments"].isna().sum()
print("подписок:", len(subs), " строк после соединения:", len(subs_pay))
answer_2 = int(no_payment)
answer_2

подписок: 25609  строк после соединения: 25609


4660

### Задача 3. Кто платил

Сколько владельцев из сегмента заплатили за полгода хотя бы раз? Для этого к каждому платежу приписывают владельца подписки. Но прежде чем соединять, объявляют кратность: сколько платежей приходится на одну подписку и сколько подписок — на один платёж.

**Что сделать**

Впишите в `validate=` кратность связи «платёж → подписка».

**Формат ответа:** `int` — например `5432`.

**Подсказка:** Слайд 6 Л4, четыре случая кратности. Если слева на один ключ приходится много строк, это «m».

In [12]:
success = pay[pay["status"] == "success"]

# Сколько платежей приходится на одну подписку — и сколько подписок на один платёж?
pay_owner = success.merge(
    subs[["subscription_id", "owner_client_id"]],
    on="subscription_id", how="left", validate="m:1",
)

payers = pay_owner["owner_client_id"].nunique()
print("платежей до соединения:", len(success), " после:", len(pay_owner))
answer_3 = int(payers)
answer_3

платежей до соединения: 108771  после: 108771


20949

### Задача 4. Выручка без канала

Маркетинг смотрит выручку по каналам привлечения. У части клиентов канал не записан, и если этого не заметить, сумма по каналам тихо разойдётся с общей выручкой.

**Что сделать**

Сделайте так, чтобы клиенты без канала попали в отчёт отдельной строкой, и посчитайте их выручку.

**Формат ответа:** `float` до двух знаков, рубли — например `123456.7`. Если таких клиентов в сегменте нет — `0.0`.

**Подсказка:** Слайд 18 Л4: сумма по группам не сходится с общей. Сверьте сумму по каналам с выручкой задачи 1.

In [13]:
success = pay[pay["status"] == "success"]
pay_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="m:1"))

# Клиенты без канала должны остаться в отчёте отдельной строкой.
by_channel = pay_client.groupby("acquisition_channel", dropna=False)["amount"].sum()

print(by_channel.round(2).to_string())
no_channel = by_channel[by_channel.index.isna()].sum()
answer_4 = round(float(no_channel), 2)
answer_4

acquisition_channel
crm                 4315715.0
organic            10105095.0
partner_telecom     5895259.0
performance         5817648.0
referral            3104353.0
social              3365952.0
NaN                  671857.0


671857.0

**Вопрос 1.** В `pay` есть попытки списания со статусом `refunded`, и сумма у них положительная. Сколько таких строк в вашем сегменте и на какую сумму? Почему эту сумму нельзя прибавить к выручке — и почему её нельзя вычесть из выручки?

In [27]:
# Числа для ответа на вопрос 1 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
refunded = pay[
    (pay["status"] == "refunded") & (pay["amount"] > 0)
]
volume_refunded = len(refunded)
sum_refunded = round(float(refunded['amount'].sum()),2)
print(f"Количество случаев со списанием в статусе возвращено:{volume_refunded}")
print(f"Сумма:{sum_refunded}")

Количество случаев со списанием в статусе возвращено:1378
Сумма:423322.0


#### ✍️ Ответ на вопрос 1
Таким образом, 1378 случаев со списанием средств в статусе "возвращено" на сумму 423 322 руб. Данная сумма возвращена клиентам, поэтому она не может являться доходом для компании. Но также и не нужно вычитать из выручки, так как при подсчитывании выручки были уже отобраны операции с успешными платежами, возвраты исключены.

## Часть 3. Уровень 2 — дыра в строку

Теперь не хватает целой строки: выражения, условия или набора аргументов. Всё остальное готово. Правила те же: строки с `answer_N` не трогайте, не получилось — `answer_N = None`.

### Задача 5. Свернуть до клиента

Средний доход с платящего клиента — ARPPU — это выручка, делённая на число клиентов, которые платили. Чтобы посчитать его по витрине, платежи сначала сворачивают до зерна «клиент» и только потом соединяют с клиентами — иначе соединение перемножит строки.

**Что сделать**

Допишите `agg(...)`: для каждого владельца — `n_payments` (сколько успешных платежей) и `paid` (сколько он заплатил).

**Формат ответа:** `float` до двух знаков, рубли — например `1234.56`.

**Подсказка:** Слайд 17 Л4, именованные агрегаты: `agg(имя=("колонка", "функция"), ...)`.

In [38]:
success = pay[pay["status"] == "success"]

# Одна строка — один владелец: сколько успешных платежей и сколько денег.
by_client = (success
             .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
             .groupby("owner_client_id", as_index=False)
             .agg(n_payments=('payment_id','size'),paid=('amount','sum')))

mart5 = clients.merge(by_client, left_on="client_id", right_on="owner_client_id",
                      how="left", validate="1:1")

paying = mart5["n_payments"].notna().sum()
arppu = mart5["paid"].sum() / paying
print("клиентов:", len(mart5), " из них платили:", paying)
answer_5 = round(float(arppu), 2)
answer_5

клиентов: 25609  из них платили: 20949


1588.42

### Задача 6. Цена по справочнику

В `tariffs` лежит история цен: у тарифа несколько строк с интервалами действия, а одна строка ошибочная — вы нашли её на семинаре. Сколько принёс тариф «Расширенный» по цене справочника?

**Что сделать**

Допишите две строки: 1) справочник без ошибочной строки; 2) условие «момент платежа попадает в интервал действия цены» по конвенции курса.

**Формат ответа:** `float` до двух знаков, рубли — например `2345678.9`.

**Подсказка:** Раздел 2 семинара С4. Конвенция границы — `(paid_at >= valid_from) & (paid_at < valid_to + 1 день)`, потому что `valid_to` — это дата, то есть полночь. Каждый платёж должен найти в справочнике ровно одну цену: сравните число строк до и после.

In [39]:
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
ref_fixed = ref[ref['monthly_fee'] != 429].copy()

pay_t = (pay[pay["status"] == "success"]
         .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
m = pay_t.merge(ref_fixed[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]],
                on="tariff_id")

# 2. Момент платежа попадает в интервал действия цены — по конвенции курса.
hit = ((m['paid_at'] >= m['valid_from']) & (m['paid_at'] < m['valid_to'] + pd.Timedelta(days=1)))
priced = m[hit]

print("платежей:", len(pay_t), " строк с ценой:", len(priced))
extended = priced.loc[priced["tariff_name"] == "Расширенный", "monthly_fee"].sum()
answer_6 = round(float(extended), 2)
answer_6

платежей: 108771  строк с ценой: 108771


17528201.0

### Задача 7. Выше среднего по своему каналу

Маркетинг хочет знать, сколько платящих клиентов приносят больше, чем типичный клиент **их же** канала. Для этого рядом с каждым клиентом ставят среднее по его каналу — не сворачивая таблицу.

**Что сделать**

Допишите строку: `channel_avg` — среднее `paid` по каналу клиента, по значению в каждой строке.

**Формат ответа:** `int` — например `6543`.

**Подсказка:** Слайд 19 Л4: `transform` возвращает столько же строк, сколько было.

In [41]:
success = pay[pay["status"] == "success"]
per_client = (success
              .merge(subs[["subscription_id", "owner_client_id"]], on="subscription_id", validate="m:1")
              .groupby("owner_client_id", as_index=False)
              .agg(paid=("amount", "sum"))
              .merge(clients[["client_id", "acquisition_channel"]],
                     left_on="owner_client_id", right_on="client_id", validate="1:1"))
per_client = per_client[per_client["acquisition_channel"].notna()]   # только известные каналы

# Рядом с каждым клиентом — среднее paid по его каналу. Строк остаётся столько же.
per_client["channel_avg"] = per_client.groupby('acquisition_channel')['paid'].transform('mean')

above = (per_client["paid"] > per_client["channel_avg"]).sum()
print("платящих клиентов с известным каналом:", len(per_client))
answer_7 = int(above)
answer_7

платящих клиентов с известным каналом: 20541


6915

### Задача 8. Выручка по месяцам

Финансам нужна таблица: строки — месяцы, колонки — тарифы, в клетках — выручка. По ней видно, как выручка «Базового» менялась от месяца к месяцу.

**Что сделать**

Допишите аргументы `pivot_table`: строки — `month`, колонки — `tariff_id`, значения — сумма `amount`.

**Формат ответа:** `float` до одного знака, проценты — например `87.6`.

**Подсказка:** Слайд 23 Л4.

In [42]:
success = (pay[pay["status"] == "success"]
           .merge(subs[["subscription_id", "tariff_id"]], on="subscription_id", validate="m:1"))
success["month"] = success["paid_at"].dt.strftime("%Y-%m")

# Строки — месяцы, колонки — тарифы (1 — «Базовый», 2 — «Расширенный»), в клетках — выручка.
by_month = success.pivot_table(index='month', columns='tariff_id', values='amount', aggfunc='sum')

print(by_month.round(2).to_string())
april_to_march = by_month.loc["2026-04", 1] / by_month.loc["2026-03", 1] * 100
answer_8 = round(float(april_to_march), 1)
answer_8

tariff_id          1          2
month                          
2026-01    2581030.0  3073497.0
2026-02    2253476.0  2697639.0
2026-03    2978289.0  3222024.0
2026-04    2605536.0  2799066.0
2026-05    2760165.0  2962053.0
2026-06    2569182.0  2773922.0


87.5

**Вопрос 2.** Выручка «Базового» в апреле — `answer_8` процентов от мартовской. Это отток клиентов? Проверьте по дням: сколько дней апреля есть в `pay` и как выглядит выручка по дням. Объясните, что случилось, и что вы написали бы финансам в отчёте.

In [60]:
# Числа для ответа на вопрос 2 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
april = pay[(pay["paid_at"] >= "2026-04-01") & (pay["paid_at"] < "2026-05-01")].copy()
april["day"] = april["paid_at"].dt.day
april_success = april[april["status"] == "success"].merge(
    subs[["subscription_id", "tariff_id"]],
    on="subscription_id", validate="m:1"
)
april_basic = april_success[april_success["tariff_id"] == 1]

daily_april = pd.DataFrame({
    "attempts": april.groupby("day").size(),
    "basic_revenue": april_basic.groupby("day")["amount"].sum()
}).reindex(range(1, 31)).fillna(0)
daily_april["attempts"] = daily_april["attempts"].astype(int)

print("дней апреля в pay:", april["day"].nunique())
print("дни без данных:", daily_april.index[daily_april["attempts"] == 0].tolist())
print("выручка базового в апреле от мартовской, %:", answer_8)
print(daily_april.round(2).to_string())

дней апреля в pay: 28
дни без данных: [14, 15]
выручка базового в апреле от мартовской, %: 87.5
     attempts  basic_revenue
day                         
1         634        98355.0
2         573        81174.0
3         605        88893.0
4         576        86652.0
5         578        84411.0
6         622        90636.0
7         661        97359.0
8         610        88644.0
9         596        83166.0
10        630        91632.0
11        646        95118.0
12        629        97359.0
13        646        98106.0
14          0            0.0
15          0            0.0
16        638        90885.0
17        638        91134.0
18        608        91383.0
19        611        95118.0
20        626        95118.0
21        663        98604.0
22        642        96363.0
23        595        87399.0
24        630        90885.0
25        583        93375.0
26        630        90387.0
27        746       111801.0
28        738       110307.0
29        602        89640.0
30   

#### ✍️ Ответ на вопрос 2

Выручка «Базового» в апреле составила 2 605 536 руб., или 87,5% от мартовской, но это само по себе не доказывает отток. В `pay` есть только 28 дней апреля: 14 и 15 апреля нет ни одной попытки списания, а в остальные дни выручка составляет от 81 174 до 111 801 руб. Это указывает на пропуск в данных, а не на подтвержденную потерю клиентов. В отчете финансам я отметила бы неполноту апреля и разную длину месяцев, не называл бы падение выручки оттоком и проверил бы его отдельно по датам окончания подписок.


## Часть 4. Уровень 3 — пишете сами

Заготовок больше нет: в ячейке только условие в комментарии и строка с `answer_N`. Код вы пишете целиком — от загруженных таблиц до ответа. Печатайте промежуточные результаты: так проще найти ошибку и объяснить решение на защите.

Каждая задача опирается на приём с лекции Л4 и семинара С4 — номер слайда стоит в подсказке. Ответ кладите в `answer_N` обычным `int` или `float`, а не `numpy`-числом: `int(...)`, `round(float(...), 2)`.

### Задача 9. Люди, а не места

Сколько людей сегмента пользуются ПРАЙМ? Пользуется тот, кто владеет подпиской или участвует хотя бы в одной чужой. Один человек может участвовать в нескольких подписках: если не свернуть участия до клиента, вы насчитаете места, а не людей.

**Что сделать**

Посчитайте клиентов сегмента, которые владеют подпиской (`subs`) или участвуют хотя бы в одной (`members`). Одна строка — один клиент.

**Формат ответа:** `int` — например `7654`.

**Подсказка:** Раздел 3 семинара С4: свернуть участия до клиента, соединить с `validate="1:1"`, флаг «владелец или участник».

In [66]:
# Задача 9. Сколько клиентов сегмента пользуются ПРАЙМ: владеют подпиской
# или участвуют хотя бы в одной. Одна строка — один клиент. Ответ — обычный int.
owners = (subs[['owner_client_id']].drop_duplicates().rename(columns={'owner_client_id': 'client_id'}))
owners['is_owner'] = True

members_by_client = (members.groupby('client_id', as_index=False).agg(n_memberships=('subscription_id', 'nunique')))

people = (clients[['client_id']].merge(owners, on='client_id', how='left', validate='1:1').merge(members_by_client, on='client_id', how='left', validate='1:1'))
people['n_memberships'] = people['n_memberships'].fillna(0)
people['uses_prime'] = people['is_owner'].eq(True) | (people['n_memberships'] > 0)

answer_9 = int(people['uses_prime'].sum())
answer_9

25609

### Задача 10. Сколько человек на подписку

Подпиской пользуются владелец и те, кого он подключил. Подключить можно не больше `share_slots` человек — это поле справочника тарифов. Сколько в среднем человек приходится на подписку сегмента и сколько подписок заполнены до предела?

**Что сделать**

1. Для каждой подписки из `subs`: один владелец плюс число её строк в `sub_members`.
2. Среднее по всем подпискам сегмента — первая часть ответа.
3. Присоедините к подпискам `share_slots` из `tariffs`. Подписок, где людей ровно `share_slots` + 1, — вторая часть ответа.
4. Напечатайте, у скольких подписок людей больше, чем `share_slots` + 1. Это инвариант: число обязано быть нулём.

**Формат ответа:** `[среднее, заполненных до предела]` — например `[3.21, 987]`: `float` до двух знаков и `int`.

**Подсказка:** Слайды 8 и 14 Л4. У одного `tariff_id` в справочнике несколько строк, и `validate="m:1"` об этом скажет. Какая часть справочника на самом деле нужна для `share_slots`?

In [67]:
# Задача 10. Люди на подписках сегмента, считая владельца, и лимит share_slots.
# Ответ — [среднее на подписку (float до 2 знаков), заполненных до предела (int)].
members_per_sub = (sub_members.groupby('subscription_id', as_index=False).agg(n_members=('member_id', 'size')))
slots = tariffs[['tariff_id', 'share_slots']].drop_duplicates()

sub_people = (subs[['subscription_id', 'tariff_id']].merge(members_per_sub, on='subscription_id', how='left', validate='1:1').merge(slots, on='tariff_id', how='left', validate='m:1'))
sub_people['n_members'] = sub_people['n_members'].fillna(0).astype(int)
sub_people['n_people'] = sub_people['n_members'] + 1

full_subs = int((sub_people['n_people'] == sub_people['share_slots'] + 1).sum())
over_limit = int((sub_people['n_people'] > sub_people['share_slots'] + 1).sum())

print("подписок сверх лимита:", over_limit)
answer_10 = [round(float(sub_people['n_people'].mean()), 2), full_subs]
answer_10

подписок сверх лимита: 0


[2.67, 5091]

### Задача 11. Сколько стоил клиент

CAC — сколько стоило привести одного клиента. Расходы на привлечение известны по каналу и дню, а не по клиенту: присоединить их к витрине напрямую нельзя. Поэтому CAC считают на его собственном зерне — «канал × месяц» — и только потом приписывают клиентам.

**Что сделать**

1. CAC канала за месяц — это расходы канала за месяц (`spend`), делённые на число всех клиентов базы, которые пришли из этого канала в этом месяце (`registrations`). Берите только полные месяцы расходов — с 2025-02 по 2026-07.
2. Каждому клиенту сегмента припишите CAC его канала в месяц его регистрации.
3. Посчитайте средний CAC по клиентам сегмента, у которых он определён, и число клиентов, у которых он не определён: канал не записан или регистрация вне этих месяцев.

**Формат ответа:** `[средний CAC, клиентов без CAC]` — например `[432.1, 876]`: `float` до двух знаков и `int`.

**Подсказка:** Слайд 28 Л4: `marketing_spend` не присоединяется к витрине — у него другое зерно. Месяц из даты: `pd.to_datetime(...).dt.strftime("%Y-%m")`. Соединение по двум колонкам: `on=["channel", "month"]`, и кратность объявите.

In [68]:
# суммируем расходы только за полные месяцы
spend_month = spend.copy()
spend_month["month"] = pd.to_datetime(spend_month["spend_date"]).dt.strftime("%Y-%m")
spend_month = spend_month[
    (spend_month["month"] >= "2025-02") & (spend_month["month"] <= "2026-07")
]
spend_month = spend_month.groupby(["channel", "month"], as_index=False).agg(
    spend=("amount", "sum")
)

# считаем регистрации всей базы по каналу и месяцу
reg_month = registrations.copy()
reg_month["month"] = pd.to_datetime(reg_month["reg_date"]).dt.strftime("%Y-%m")
reg_month = reg_month.groupby(["channel", "month"], as_index=False).agg(
    n_clients=("n_clients", "sum")
)
cac_month = spend_month.merge(reg_month, on=["channel", "month"], how="left", validate="1:1")
cac_month["cac"] = cac_month["spend"] / cac_month["n_clients"]

# приписываем клиенту стоимость привлечения в его месяц регистрации
client_cac = clients[["client_id", "acquisition_channel", "registered_at"]].copy()
client_cac["month"] = pd.to_datetime(client_cac["registered_at"]).dt.strftime("%Y-%m")
client_cac = client_cac.merge(
    cac_month[["channel", "month", "cac"]],
    left_on=["acquisition_channel", "month"], right_on=["channel", "month"],
    how="left", validate="m:1"
)

print("клиентов с известным cac:", client_cac["cac"].notna().sum())
answer_11 = [round(float(client_cac["cac"].mean()), 2), int(client_cac["cac"].isna().sum())]
answer_11

клиентов с известным cac: 9342


[657.11, 16267]

### Задача 12. Лучшие регионы канала

Где продавать ПРАЙМ дальше? Маркетинг просит три лучших региона по выручке в каждом канале привлечения — и начинает с самого доходного канала.

**Что сделать**

Посчитайте выручку сегмента за окно по каналу и региону владельца; клиенты без канала не участвуют. В каждом канале отберите три региона с наибольшей выручкой. Ответ — три `region_code` самого доходного канала сегмента, от большей выручки к меньшей.

**Формат ответа:** список из трёх `int` — например `[12, 34, 56]`.

**Подсказка:** Слайд 25 Л4: «три лучших вообще» и «три лучших в каждом канале» — разные вещи.

In [69]:
# Задача 12. Три лучших региона в каждом канале; ответ — для самого доходного канала.
# Ответ — список из трёх int, от большей выручки к меньшей.
# добавляем канал и регион к выручке каждого владельца
region_pay = by_client.merge(
    clients[["client_id", "acquisition_channel", "region_code"]],
    left_on="owner_client_id", right_on="client_id", validate="1:1"
)
region_pay = region_pay[region_pay["acquisition_channel"].notna()]
region_revenue = region_pay.groupby(["acquisition_channel", "region_code"], as_index=False).agg(
    revenue=("paid", "sum")
)

# выбираем три региона внутри каждого канала
ordered_regions = region_revenue.sort_values(
    ["acquisition_channel", "revenue", "region_code"], ascending=[True, False, True]
)
top_regions = ordered_regions.groupby("acquisition_channel").head(3)
best_channel = region_revenue.groupby("acquisition_channel")["revenue"].sum().idxmax()

print(top_regions.round(2).to_string(index=False))
print("самый доходный канал:", best_channel)
answer_12 = [int(region) for region in top_regions.loc[
    top_regions["acquisition_channel"] == best_channel, "region_code"
]]
answer_12

acquisition_channel  region_code  revenue
                crm           48  74356.0
                crm           10  71082.0
                crm           63  70529.0
            organic           58 144881.0
            organic           35 144098.0
            organic           13 140032.0
    partner_telecom           54  93038.0
    partner_telecom           23  90678.0
    partner_telecom           42  89559.0
        performance            7  93946.0
        performance            5  88960.0
        performance           30  87340.0
           referral           32  60430.0
           referral           37  57090.0
           referral           30  52831.0
             social           70  68023.0
             social            2  56720.0
             social           54  55784.0
самый доходный канал: organic


[58, 35, 13]

**Вопрос 3.** Куда вы добавили бы рубль на привлечение? Сравните каналы по двум числам: CAC канала (задача 11) и выручка на клиента вашего сегмента из этого канала. Назовите канал и оба числа. Почему выручку вашего сегмента нельзя просто разделить на CAC канала — какую оговорку нужно сделать?

In [55]:
# Числа для ответа на вопрос 3 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
# учитываем всех клиентов канала, в том числе без платежей
channel_clients = client_cac.merge(
    by_client[["owner_client_id", "paid"]],
    left_on="client_id", right_on="owner_client_id", how="left", validate="1:1"
)
channel_clients["paid"] = channel_clients["paid"].fillna(0)
channel_comparison = channel_clients.groupby("acquisition_channel").agg(
    n_clients=("client_id", "size"),
    n_with_cac=("cac", "count"),
    cac=("cac", "mean"),
    revenue_per_client=("paid", "mean")
)
print(channel_comparison.round(2).to_string())

                     n_clients  n_with_cac      cac  revenue_per_client
acquisition_channel                                                    
crm                       3114        1101   216.43             1385.91
organic                   6873        2341   139.84             1470.26
partner_telecom           4625        1783   682.37             1274.65
performance               5024        1996  1495.92             1157.97
referral                  2167         714   350.72             1432.56
social                    3292        1407   796.10             1022.46


#### ✍️ Ответ на вопрос 3

Я бы направила дополнительный рубль в organic: средний CAC клиентов канала с известной стоимостью равен 139,84 руб., а выручка за полугодие на одного клиента сегмента составляет 1 470,26 руб. Среди сравниваемых каналов у него самый низкий CAC и самая высокая выручка на клиента. CAC рассчитан по расходам и регистрациям всей базы в канале и месяце и известен лишь для 2 341 из 6 873 клиентов organic, а выручка относится к нашему отобранному по платежам balance сегменту. Поэтому общую выручку нельзя делить на CAC как на общие расходы на привлечение, а сравнение выручки на клиента с CAC здесь дает лишь ориентир: состав клиентов и периоды различаются, историческая средняя стоимость не гарантирует такой же результат от нового рубля.

## Часть 5. Уровень 4 — витрина и протокол сборки

Главная задача недели. Вы собираете **витрину клиента**: одна строка — один клиент сегмента, рядом всё, что про него известно. И доказываете, что двойного счёта нет, **протоколом сборки** — той же формы, что на слайде 15 Л4 и на семинаре С4.

> **Протокол — это 3 балла из 10.** Без него ноль, даже если все числа верные: результат, который нельзя перепроверить, в работе не принимают.

Три последние строки протокола — инварианты витрины:

- **ЗЕРНО:** строк в витрине столько же, сколько уникальных клиентов, и столько, сколько вы объявили до сборки;
- **ДЕНЬГИ:** сумма по витрине сходится с независимым контролем до копейки;
- **ЛЮДИ:** число тех, кто пользуется ПРАЙМ, сходится со счётом, сделанным другим путём.

Ниже готовые функции, эту ячейку просто запустите. `step()` — та же, что на семинаре: одна строка протокола на одно соединение. `check_step()` делает строку протокола для проверки из вашего `src/prime/checks.py`. `print_protocol()` печатает протокол и возвращает ответ задачи 13.

In [56]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки.

    keys_lost — строки слева, которым не нашлось пары справа.
    Видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost, "verdict": verdict}


def check_step(name: str, frame: pd.DataFrame, bad: pd.DataFrame, expected: int) -> dict:
    """Строка протокола для проверки из checks.py.

    bad — строки, которые нашла проверка; expected — сколько вы ждали, объявив это ДО сборки.
    """
    found = len(bad)
    verdict = f"ok, ожидали {expected}" if found == expected else f"ВНИМАНИЕ: нашлось {found}, ждали {expected}"
    return {"step": name, "rows_before": len(frame), "rows_after": len(frame),
            "validate": "проверка", "keys_lost": found, "verdict": verdict}


def print_protocol(protocol: list[dict], mart: pd.DataFrame, declared: int,
                   money_mart: float, money_control: float,
                   people_mart: int, people_control: int) -> list[int]:
    """Напечатать протокол сборки. Вернуть [строк в витрине, разница ДЕНЬГИ в копейках, ЛЮДИ по витрине]."""
    rows, unique = len(mart), int(mart["client_id"].nunique())
    kopecks_mart, kopecks_control = round(money_mart * 100), round(money_control * 100)
    diff = kopecks_mart - kopecks_control

    print("ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента")
    print()
    print(pd.DataFrame(protocol).to_string(index=False))
    print()
    print(f"ЗЕРНО   строк {rows}, клиентов {unique}, объявляли {declared}"
          f"   -> {'ok' if rows == unique == declared else 'ВНИМАНИЕ'}")
    print(f"ДЕНЬГИ  по витрине {money_mart:.2f} ₽, независимо {money_control:.2f} ₽, разница {diff} коп."
          f"   -> {'ok' if diff == 0 else 'ВНИМАНИЕ'}")
    print(f"ЛЮДИ    по витрине {people_mart}, независимо {people_control}"
          f"   -> {'ok' if people_mart == people_control else 'ВНИМАНИЕ'}")
    print()
    print("HW2_PROTOCOL=" + json.dumps({
        "steps": protocol, "grain": [rows, unique, declared],
        "money_kopecks": [kopecks_mart, kopecks_control], "people": [int(people_mart), int(people_control)],
    }, ensure_ascii=False, default=str))
    return [int(rows), int(diff), int(people_mart)]


print("функции протокола готовы")

функции протокола готовы


### Задача 13. Витрина клиента

Соберите витрину по своему сегменту и докажите протоколом, что двойного счёта нет. Весь раздел 4 семинара С4 — это тот же путь на общем сегменте.

**Что сделать**

1. **До сборки** объявите, сколько строк будет в витрине, и сколько ключей потеряет каждое левое соединение — эти числа пойдут в `expect_lost`.
2. Соберите `mart`, одна строка — один клиент сегмента:
   - `client_id`, `acquisition_channel`, `region_code` — из `clients`;
   - `subscription_id`, `tariff_id` — подписка клиента, пропуск, если её нет;
   - `n_payments`, `paid` — успешные платежи за окно, свёрнутые до клиента; `0`, если их нет;
   - `n_memberships` — в скольких подписках клиент участник; `0`, если ни в одной;
   - `uses_prime` — владелец или участник.
3. Каждое соединение запишите в протокол через `step()`.
4. Примените к витрине функцию из своего `src/prime/checks.py` — например, `find_missing_channel` — и запишите её строкой `check_step()` с ожиданием, объявленным до сборки. Это отдельные 0,5 балла из 10.
5. Посчитайте независимые контроли для ДЕНЬГИ и ЛЮДИ **не по витрине, а другим путём** и вызовите `print_protocol()`.

**Формат ответа:** то, что вернула `print_protocol()`: список из трёх `int`.

**Подсказка:** Если `src/prime/checks.py` у вас нет — создайте его по образцу семинара С3 и закоммитьте в `main`: `from prime.checks import find_missing_channel`. Протокол должен уметь покраснеть: проверьте его, сломав одно соединение, как на семинаре.

In [62]:
# Задача 13. Витрина клиента и протокол сборки.
# Одна строка — один клиент сегмента. Каждое соединение — через step()
# с ожиданием, объявленным ДО сборки. Проверка из checks.py — через check_step().
# В конце — answer_13 = print_protocol(...).

from prime.checks import find_missing_channel

# объявляем число строк и ожидаемые потери до сборки
expected_rows = len(clients)
expected_subs = int((~clients["client_id"].isin(subs["owner_client_id"])).sum())
paid_owners = subs.loc[
    subs["subscription_id"].isin(pay.loc[pay["status"] == "success", "subscription_id"]),
    "owner_client_id"
]
expected_payments = int((~clients["client_id"].isin(paid_owners)).sum())
expected_members = int((~clients["client_id"].isin(members["client_id"])).sum())
expected_missing_channel = int(clients["acquisition_channel"].isna().sum())

# считаем независимые контроли по исходным таблицам
money_control = float(pay.loc[pay["status"] == "success", "amount"].sum())
people_control = len(set(subs["owner_client_id"]) | set(members["client_id"]))

protocol = []
mart = clients[["client_id", "acquisition_channel", "region_code"]].copy()

# добавляем подписку, сохраняя клиентов без подписки
before = mart
mart = mart.merge(
    subs[["owner_client_id", "subscription_id", "tariff_id"]].rename(
        columns={"owner_client_id": "client_id"}
    ),
    on="client_id", how="left", validate="1:1"
)
protocol.append(step("подписки", before, mart, "subscription_id", "1:1", expected_subs))

# платежи уже свернуты до клиента в задаче 5
before = mart
mart = mart.merge(
    by_client.rename(columns={"owner_client_id": "client_id"}),
    on="client_id", how="left", validate="1:1"
)
protocol.append(step("платежи", before, mart, "n_payments", "1:1", expected_payments))

# участия уже свернуты до клиента в задаче 9
before = mart
mart = mart.merge(members_by_client, on="client_id", how="left", validate="1:1")
protocol.append(step("участия", before, mart, "n_memberships", "1:1", expected_members))

mart["n_payments"] = mart["n_payments"].fillna(0).astype(int)
mart["paid"] = mart["paid"].fillna(0)
mart["n_memberships"] = mart["n_memberships"].fillna(0).astype(int)
mart["uses_prime"] = mart["subscription_id"].notna() | (mart["n_memberships"] > 0)

# записываем проверку из отдельного файла
bad = find_missing_channel(mart)
protocol.append(check_step("нет канала", mart, bad, expected_missing_channel))

answer_13 = print_protocol(
    protocol, mart, expected_rows,
    float(mart["paid"].sum()), money_control,
    int(mart["uses_prime"].sum()), people_control
)
answer_13

ПРОТОКОЛ СБОРКИ ВИТРИНЫ: одна строка — один клиент сегмента

      step  rows_before  rows_after validate  keys_lost          verdict
  подписки        25609       25609      1:1          0    ok, ожидали 0
   платежи        25609       25609      1:1       4660 ok, ожидали 4660
   участия        25609       25609      1:1       8370 ok, ожидали 8370
нет канала        25609       25609 проверка        514  ok, ожидали 514

ЗЕРНО   строк 25609, клиентов 25609, объявляли 25609   -> ok
ДЕНЬГИ  по витрине 33275879.00 ₽, независимо 33275879.00 ₽, разница 0 коп.   -> ok
ЛЮДИ    по витрине 25609, независимо 25609   -> ok

HW2_PROTOCOL={"steps": [{"step": "подписки", "rows_before": 25609, "rows_after": 25609, "validate": "1:1", "keys_lost": 0, "verdict": "ok, ожидали 0"}, {"step": "платежи", "rows_before": 25609, "rows_after": 25609, "validate": "1:1", "keys_lost": 4660, "verdict": "ok, ожидали 4660"}, {"step": "участия", "rows_before": 25609, "rows_after": 25609, "validate": "1:1", "keys_lost

[25609, 0, 25609]

**Вопрос 4.** Где ваш протокол заранее ждал потерь ключей и почему это не ошибка — назовите числа. Что стало бы со строками ЗЕРНО и ЛЮДИ, если бы участия не свернули до клиента? Посчитайте, а не угадывайте.

In [63]:
# Числа для ответа на вопрос 4 считайте здесь — по загруженным таблицам.
# Ячейка тоже выполняется на контрольном сегменте: id и числа руками не вписывайте.
print("ожидали клиентов без подписки:", expected_subs)
print("ожидали клиентов без успешных платежей:", expected_payments)
print("ожидали клиентов без участий:", expected_members)

# намеренно присоединяем участия без свертки
wrong_before = mart[["client_id", "subscription_id"]]
wrong_mart = wrong_before.merge(
    members.rename(columns={"subscription_id": "member_subscription_id"}),
    on="client_id", how="left", validate="1:m"
)
wrong_mart["uses_prime"] = (
    wrong_mart["subscription_id"].notna() | wrong_mart["member_subscription_id"].notna()
)
print("зерно: верно", len(mart), "; без свертки", len(wrong_mart),
      "; уникальных клиентов", wrong_mart["client_id"].nunique())
print("люди: верно", int(mart["uses_prime"].sum()),
      "; без свертки", int(wrong_mart["uses_prime"].sum()))

# проверяем, что протокол замечает лишние строки
wrong_step = step(
    "участия без свертки", wrong_before, wrong_mart,
    "member_subscription_id", "1:m", expected_members
)
print(pd.DataFrame([wrong_step]).to_string(index=False))

ожидали клиентов без подписки: 0
ожидали клиентов без успешных платежей: 4660
ожидали клиентов без участий: 8370
зерно: верно 25609 ; без свертки 36893 ; уникальных клиентов 25609
люди: верно 25609 ; без свертки 36893
               step  rows_before  rows_after validate  keys_lost                                 verdict
участия без свертки        25609       36893      1:m       8370 ВНИМАНИЕ: строк было 25609, стало 36893


#### ✍️ Ответ на вопрос 4

До сборки я ожидала 0 клиентов без подписки, 4 660 без успешных платежей за окно и 8 370 без участий. Это не потеря клиентов: при левом соединении их строки остаются, а отсутствие платежей и участий заменяется нулями; отдельная проверка также нашла ожидаемые 514 клиентов без канала. Без свертки участий получилось 36 893 строки при 25 609 уникальных клиентах, поэтому нарушилось бы ЗЕРНО. ЛЮДИ тоже выросли бы с 25 609 до 36 893, то есть были бы завышены на 11 284; протокол показал предупреждение о росте числа строк.

## Часть 6. Итог — готовый код

При **Restart Kernel and Run All Cells** эта ячейка выполняется после всех задач: перед сдачей посмотрите её вывод. Она показывает ваши ответы и проверяет только формат, но не правильность.

Последняя строка вывода служебная: по ней преподаватель сверяет ответы автоматически. Ячейку не удаляйте.

In [ ]:
def typed(x):
    """Для служебной строки: всё, кроме list/dict/str/int/float/bool/None, помечаем типом."""
    if type(x) is dict:
        return {str(key): typed(value) for key, value in x.items()}
    if type(x) is list:
        return [typed(value) for value in x]
    if x is None or type(x) in (bool, int, float, str):
        return x
    return f"<{type(x).__name__}> {x!r}"


answers = {n: globals().get(f"answer_{n}") for n in range(1, 14)}
formats = {n: format_problem(n, answer) for n, answer in answers.items()}

for n, answer in answers.items():
    print(f"задача {n:>2}: {answer!r}")
    print("    ✅ формат в порядке" if formats[n] is None else f"    ❌ {formats[n]}")

print("\nНе забудьте ответы словами — ячейки «✍️ Ответ на вопрос N» после каждого уровня.")
print("\n--- служебная строка, не удаляйте ---")
print("HW2_ANSWERS=" + json.dumps(
    {"client_filter": CLIENT_FILTER, "answers": typed(answers), "format": formats}, ensure_ascii=False
))

## Часть 7. Исследование — для тех, кто хочет 9 или 10

**Эта часть необязательна.** Части 1–6 — это оценка до 8 баллов, и 8 — это «отлично». 9 и 10 ставятся только за исследование: в нём две части, каждая до 1 балла сверху. Они засчитываются, если основная часть набрала не меньше 15 баллов из 20.

Здесь нет готового формата ответа и заготовок. Вы сами решаете, что посчитать, считаете это кодом и пишете вывод. Ассистент может помочь с кодом, но ответ зависит от того, что вы увидите в своих данных. Текст вывода — ваш: правило про LLM действует и здесь.

| Каждая часть оценивается так | Балл |
|---|---|
| числа верные и получены кодом в ноутбуке, а не вписаны руками | 0,4 |
| вывод следует из чисел, и названо, чего ваш расчёт **не** доказывает | 0,3 |
| код выполняется у преподавателя целиком — в том числе на контрольном сегменте | 0,2 |
| изложено по схеме: вопрос → как проверяли → что получилось → что это значит | 0,1 |

Загруженных таблиц для исследования хватает: `pay`, `subs` и `tariffs`. Считайте от них, а не от чисел, вписанных руками, — на контрольном сегменте код должен отработать без правок.

### Исследование А. Что подорожание дало выручке

С 1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽. Финансы сравнили выручку сегмента за март–июнь с январём–февралём и записали весь рост в заслугу подорожания. Руководитель просит проверить, сколько рублей в месяц на самом деле дала цена.

**Что сделать**

1. По месяцам окна посчитайте выручку и число успешных платежей — по каждому тарифу.
2. Разделите изменение выручки после 1 марта на две части. **Цена** — разница между тем, сколько стоили платежи марта–июня, и тем, сколько те же платежи стоили бы по старой цене. **Объём** — всё остальное.
3. Учтите две вещи, которые меняют число платежей в месяц без всякого подорожания. Подписка списывается раз в 30 дней, а месяцы разной длины. И в данных апреля есть дыра — вы видели её в вопросе 2.
4. Ответьте: сколько рублей в месяц приносит подорожание вашему сегменту — и насколько ошибается наивное сравнение «март–июнь против января–февраля».

In [71]:
# Исследование А: ваш код. Всё — от загруженных таблиц.
# находим старые цены в исправленном справочнике
old_prices = ref_fixed[
    (ref_fixed["valid_from"] <= pd.Timestamp(WINDOW[0], tz="UTC"))
    & (ref_fixed["valid_to"] >= pd.Timestamp(WINDOW[0], tz="UTC"))
][["tariff_id", "monthly_fee"]].rename(columns={"monthly_fee": "old_fee"})

# оцениваем те же успешные платежи по старой цене
price_pay = priced[["payment_id", "paid_at", "tariff_id", "amount"]].merge(
    old_prices, on="tariff_id", validate="m:1"
)
price_pay["month"] = price_pay["paid_at"].dt.strftime("%Y-%m")
price_pay["price_effect"] = price_pay["amount"] - price_pay["old_fee"]
monthly_price = price_pay.groupby(["month", "tariff_id"], as_index=False).agg(
    n_payments=("payment_id", "size"),
    revenue=("amount", "sum"),
    old_revenue=("old_fee", "sum"),
    price_effect=("price_effect", "sum")
)

# оставляем в таблице все шесть месяцев и оба тарифа
month_starts = pd.date_range(WINDOW[0], WINDOW[1], freq="MS", inclusive="left", tz="UTC")
calendar = pd.DataFrame({
    "month": month_starts.strftime("%Y-%m"),
    "calendar_days": month_starts.days_in_month
})
tariff_names = ref_fixed[["tariff_id", "tariff_name"]].drop_duplicates()
monthly_price = calendar.merge(tariff_names, how="cross").merge(
    monthly_price, on=["month", "tariff_id"], how="left", validate="1:1"
)
value_columns = ["n_payments", "revenue", "old_revenue", "price_effect"]
monthly_price[value_columns] = monthly_price[value_columns].fillna(0)
monthly_price["n_payments"] = monthly_price["n_payments"].astype(int)

# считаем дни с данными по всем статусам, а не только по успешным платежам
coverage = pd.DataFrame({"day": pay["paid_at"].dt.normalize().drop_duplicates()})
coverage["month"] = coverage["day"].dt.strftime("%Y-%m")
coverage = coverage.groupby("month", as_index=False).agg(observed_days=("day", "nunique"))
monthly_price = monthly_price.merge(coverage, on="month", how="left", validate="m:1")
monthly_price["observed_days"] = monthly_price["observed_days"].fillna(0).astype(int)
print("помесячные результаты:")
print(monthly_price.round(2).to_string(index=False))

# раскладываем наивную разницу среднемесячной выручки на цену и объем
before_price = monthly_price[monthly_price["month"] < "2026-03"].groupby("tariff_id")[value_columns].mean()
after_price = monthly_price[monthly_price["month"] >= "2026-03"].groupby("tariff_id")[value_columns].mean()
raw_effect = pd.DataFrame({
    "naive_change": after_price["revenue"] - before_price["revenue"],
    "price": after_price["price_effect"],
    "volume": after_price["old_revenue"] - before_price["revenue"]
})
raw_effect.loc["всего"] = raw_effect.sum()
print("разложение по наблюдаемым календарным месяцам, руб. в месяц:")
print(raw_effect.round(2).to_string())

# исключаем апрель с пропуском, не восстанавливая отсутствующие платежи
complete = monthly_price[monthly_price["month"] != "2026-04"].copy()
complete["period"] = np.where(complete["month"] < "2026-03", "до", "после")
period_price = complete.groupby(["tariff_id", "period"]).agg(
    days=("calendar_days", "sum"),
    n_payments=("n_payments", "sum"),
    revenue=("revenue", "sum"),
    old_revenue=("old_revenue", "sum"),
    price_effect=("price_effect", "sum")
)

# приводим полные месяцы к одинаковому периоду в 30 дней
for column in value_columns:
    period_price[column + "_30"] = period_price[column] / period_price["days"] * 30
before_30 = period_price.xs("до", level="period")
after_30 = period_price.xs("после", level="period")
corrected_effect = pd.DataFrame({
    "payments_before_30": before_30["n_payments_30"],
    "payments_after_30": after_30["n_payments_30"],
    "revenue_before_30": before_30["revenue_30"],
    "revenue_after_30": after_30["revenue_30"],
    "change_30": after_30["revenue_30"] - before_30["revenue_30"],
    "price_30": after_30["price_effect_30"],
    "volume_30": after_30["old_revenue_30"] - before_30["revenue_30"]
})
corrected_effect["naive_error"] = raw_effect.loc[corrected_effect.index, "naive_change"] - corrected_effect["price_30"]
corrected_effect.loc["всего"] = corrected_effect.sum()
print("исключен апрель 2026 из-за пропуска в данных")
print("результат на 30 дней; naive_error = наивный рост минус эффект цены:")
print(corrected_effect.round(2).to_string())

помесячные результаты:
  month  calendar_days  tariff_id tariff_name  n_payments   revenue  old_revenue  price_effect  observed_days
2026-01             31          1     Базовый       12970 2581030.0    2581030.0           0.0             31
2026-01             31          2 Расширенный        7703 3073497.0    3073497.0           0.0             31
2026-02             28          1     Базовый       11324 2253476.0    2253476.0           0.0             28
2026-02             28          2 Расширенный        6761 2697639.0    2697639.0           0.0             28
2026-03             31          1     Базовый       11961 2978289.0    2380239.0      598050.0             31
2026-03             31          2 Расширенный        7176 3222024.0    2863224.0      358800.0             31
2026-04             30          1     Базовый       10464 2605536.0    2082336.0      523200.0             28
2026-04             30          2 Расширенный        6234 2799066.0    2487366.0      311700.0   

#### ✍️ Исследование А

Проверяла, сколько в изменении выручки связано с повышением цены, а сколько — с числом платежей. Для каждого тарифа посчитала успешные платежи и выручку по месяцам, затем оценила те же платежи по старым ценам. Разницу отнесла к цене, остальное изменение выручки — к объему.

Наивное сравнение средних за календарный месяц дает рост на 364 738,25 руб. Это сумма ценового эффекта +875 162,50 руб. и изменения объема −510 424,25 руб. Поэтому весь наблюдаемый рост нельзя считать вкладом цены: в этом сегменте снижение числа платежей скрывает часть прибавки от подорожания.

Списания происходят раз в 30 дней, а месяцы имеют разную длину. Кроме того, в апреле отсутствуют данные за 14 и 15 числа. Поэтому для сопоставимого расчета я исключила апрель, не восстанавливала пропущенные платежи и привела январь–февраль и март, май, июнь к 30 дням. При наблюдаемом числе платежей повышение цены дает 543 978,26 руб. на «Базовом» и 325 288,04 руб. на «Расширенном», всего **869 266,30 руб. за 30 дней**. Изменение объема составляет −631 867,26 руб., а итоговый рост выручки — 237 399,05 руб. за 30 дней. Наивная прибавка 364 738,25 руб. занижает нормированный вклад цены на **504 528,05 руб.**

Это расчет при том же числе и составе успешных платежей, а не доказательство полного причинного эффекта подорожания. Мы не знаем, сколько платежей было бы без изменения цены. Приведение к 30 дням учитывает длину месяцев, но не убирает все колебания из-за дат списаний. Исключение апреля устраняет влияние известного пропуска на эту оценку, но не восстанавливает фактическую выручку за потерянные дни.


### Исследование Б. Стоило ли подорожание подписчиков

Маркетинг опасается, что подорожание вызвало волну отмен, и предлагает вернуть старые цены. Проверьте на своём сегменте, есть ли для этого основания.

**Что сделать**

1. По месяцам с января по июнь посчитайте отток: сколько подписок закончилось в месяце из тех, что были активны на его начало. Данные — `subs`, колонки `started_at` и `ended_at`.
2. Сравните отток до и после 1 марта — отдельно для «Базового» (+25 %) и «Расширенного» (+12,5 %). Если бы цена отпугивала, какой тариф должен был отреагировать сильнее?
3. Посмотрите на причины отмен: как менялась доля `too_expensive` среди отмен по месяцам.
4. Вывод для маркетинга: есть ли в данных след подорожания в оттоке — и чего ваш расчёт не доказывает. Сколько у вас отмен в месяц и хватит ли этого, чтобы заметить эффект?

In [72]:
# считаем отмены только среди активных на начало месяца
churn_parts = []
for month in pd.date_range(WINDOW[0], WINDOW[1], freq="MS", inclusive="left", tz="UTC"):
    next_month = month + pd.DateOffset(months=1)
    active = subs[
        (subs["started_at"] <= month)
        & (subs["ended_at"].isna() | (subs["ended_at"] > month))
    ].copy()
    active["churned"] = active["ended_at"] < next_month
    counts = active.groupby("tariff_id").agg(
        active_start=("subscription_id", "size"),
        churned=("churned", "sum")
    ).reindex(tariffs["tariff_id"].unique(), fill_value=0)
    counts["month"] = month.strftime("%Y-%m")
    churn_parts.append(counts.reset_index())
churn_month = pd.concat(churn_parts, ignore_index=True)
churn_month["churn_pct"] = churn_month["churned"] / churn_month["active_start"] * 100

# причины смотрим среди всех отмен месяца, включая новые подписки
cancelled = subs[
    (subs["ended_at"] >= WINDOW[0]) & (subs["ended_at"] < WINDOW[1])
].copy()
cancelled["month"] = cancelled["ended_at"].dt.strftime("%Y-%m")
cancelled["too_expensive"] = cancelled["cancel_reason"] == "too_expensive"
reasons = cancelled.groupby(["month", "tariff_id"], as_index=False).agg(
    cancellations=("subscription_id", "size"),
    too_expensive=("too_expensive", "sum")
)
churn_month = churn_month.merge(reasons, on=["month", "tariff_id"], how="left", validate="1:1")
churn_month[["cancellations", "too_expensive"]] = churn_month[
    ["cancellations", "too_expensive"]
].fillna(0).astype(int)
churn_month["too_expensive_pct"] = churn_month["too_expensive"] / churn_month["cancellations"] * 100
print("отток и причины отмен по месяцам, доли в процентах:")
print(churn_month.round(3).to_string(index=False))

# сравниваем периоды с учетом числа активных подписок
churn_month["period"] = np.where(churn_month["month"] < "2026-03", "до", "после")
churn_period = churn_month.groupby(["tariff_id", "period"]).agg(
    active_start=("active_start", "sum"),
    churned=("churned", "sum"),
    cancellations=("cancellations", "sum"),
    too_expensive=("too_expensive", "sum"),
    n_months=("month", "size")
)
churn_period["churn_pct"] = churn_period["churned"] / churn_period["active_start"] * 100
churn_period["too_expensive_pct"] = churn_period["too_expensive"] / churn_period["cancellations"] * 100
churn_period["cancellations_per_month"] = churn_period["cancellations"] / churn_period["n_months"]
print("сравнение до и после 1 марта:")
print(churn_period.round(3).to_string())

# показываем изменение оттока и размер шага в одну отмену
churn_before = churn_period.xs("до", level="period")
churn_after = churn_period.xs("после", level="period")
churn_change = pd.DataFrame({
    "before_pct": churn_before["churn_pct"],
    "after_pct": churn_after["churn_pct"],
    "change_pp": churn_after["churn_pct"] - churn_before["churn_pct"],
    "reason_change_pp": churn_after["too_expensive_pct"] - churn_before["too_expensive_pct"]
})
print("изменения долей, процентные пункты:")
print(churn_change.round(3).to_string())
churn_month["one_cancel_pp"] = 100 / churn_month["active_start"]
churn_month["one_reason_pp"] = 100 / churn_month["cancellations"]
print("влияние одной отмены на месячные доли, процентные пункты:")
print(churn_month[["month", "tariff_id", "one_cancel_pp", "one_reason_pp"]].round(3).to_string(index=False))

отток и причины отмен по месяцам, доли в процентах:
 tariff_id  active_start  churned   month  churn_pct  cancellations  too_expensive  too_expensive_pct
         1         13148      509 2026-01      3.871            509            110             21.611
         2          7826      296 2026-01      3.782            296             89             30.068
         1         12639      480 2026-02      3.798            480            123             25.625
         2          7530      274 2026-02      3.639            274             62             22.628
         1         12159      467 2026-03      3.841            467            117             25.054
         2          7256      269 2026-03      3.707            269             63             23.420
         1         11692      442 2026-04      3.780            442             89             20.136
         2          6987      266 2026-04      3.807            266             74             27.820
         1         11250      

#### ✍️ Исследование Б
Проверяла, появилась ли после подорожания волна отмен. По каждому месяцу и тарифу посчитала долю закончившихся подписок среди активных на начало месяца. Отдельно рассчитала долю `too_expensive` среди всех отмен месяца. При сравнении периодов делила суммарные отмены на сумму чисел активных подписок на начало месяцев, поэтому это взвешенный месячный отток, а не доля отмен за весь период.

При одинаковой реакции на относительное изменение цены сильнее должен был отреагировать «Базовый»: он подорожал примерно на 25%, а «Расширенный» — на 12,5%. Но отток «Базового» изменился с 3,835% до 3,697%, то есть снизился на 0,138 процентного пункта. У «Расширенного» он почти не изменился: 3,712% до и 3,706% после. В марте скачка нет: отток составил 3,841% и 3,707% соответственно. Доля `too_expensive` у «Базового» практически прежняя: 23,559% до и 23,616% после; у «Расширенного» она снизилась с 26,491% до 23,599%.

В месяц наблюдается 366–509 отмен «Базового» и 241–296 отмен «Расширенного». После повышения это в среднем 424,5 и 254,25 отмены в месяц соответственно. Отмен сотни, поэтому крупную устойчивую волну можно было бы заметить, но небольшой эффект этим сравнением надежно исключить нельзя. Шаг доли в одну отмену, показанный в коде, не является оценкой случайной погрешности или доказательством статистической значимости.

По этому сегменту я не вижу оснований возвращать старые цены именно из-за предполагаемой волны отмен: ни отток, ни доля причины `too_expensive` не показывают устойчивого роста после 1 марта. Это не доказывает, что цена совсем не влияет на отмены. Оба тарифа подорожали, группы с неизменной ценой нет, а со временем меняется состав оставшихся подписчиков. Сегмент отобран по платежам `balance` за прошлый период, поэтому вывод нельзя автоматически переносить на всю базу. Пропуск в апрельских платежах не требует исключать апрель здесь: отток рассчитан по датам в `subs`, а не по `pay`.
